# Microkinetic Modeling of ORR on FePc

## Why this matters

Thermodynamic diagrams identify favorable states and equilibrium coverage,
but kinetics and mass transport determine measurable current.  A
microkinetic model connects those layers in one internally consistent
framework.

You will solve steady-state coverages, calculate kinetic and
diffusion-limited currents, locate the half-wave potential, and explore
parameter sensitivity with interactive controls.

## Learning objectives

By the end of this notebook, you should be able to:

- Translate an ORR free-energy landscape into reversible elementary rates.
- Solve steady-state coverages with a site-balance constraint.
- Combine kinetic and Levich currents with the Koutecky–Levich relation.
- Identify thermodynamic, kinetic, site-density, and transport control.


In [7]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve
from ipywidgets import FloatSlider, FloatLogSlider, HBox, VBox, interactive_output
from IPython.display import display
right_interact = lambda function, **controls: display(HBox([interactive_output(function, controls), VBox(list(controls.values()))]))

In [8]:
plt.rc("figure", figsize=(8.25/2.54, 8.25/2.54))
plt.rc("figure", dpi=100)
plt.rc("lines",  linewidth=4)
plt.rc("font",   size=10)
plt.rc("font",   family="sans-serif")
plt.rc("axes",   titlesize=10)
plt.rc("axes",   labelsize=9)
plt.rc("xtick",  labelsize=8)
plt.rc("ytick",  labelsize=8)
plt.rc("legend", fontsize=8)
plt.rc("figure", titlesize=10)
colors = ["#1965B0", "#7BAFDE", "#90C987", "#F1932D", "#DC050C"]

## Microkinetics: minimal theory

A microkinetic model starts from an explicit list of elementary reactions.
It assigns a forward and reverse rate to each step, then solves the coupled
material balances without assuming a single rate-determining step.

For a reversible elementary step $i$, mean-field mass action gives

$$
r_i=r_i^+-r_i^-
   =k_i^+\prod_m a_m^{\nu_{mi}^{\mathrm{react}}}
   -k_i^-\prod_n a_n^{\nu_{ni}^{\mathrm{prod}}}.
$$

The activity of an adsorbed species is approximated by its coverage. For a
simple step $\mathrm{A*\rightleftharpoons B*}$,

$$
r_i=k_i^+\theta_{\mathrm{A}}-k_i^-\theta_{\mathrm{B}}.
$$

Thermodynamic consistency constrains the ratio of the rate constants through
**detailed balance**:

$$
\frac{k_i^+}{k_i^-}=K_i
  =\exp\left(-\frac{\Delta G_i^0}{k_BT}\right),
$$

after activities and standard states are defined consistently. A
free-energy diagram therefore fixes the equilibrium tendency of a step, but
it does **not** fix its absolute speed. An activation barrier or standard
rate constant is still required.

For a one-electron step, this notebook uses a Butler–Volmer-type potential
dependence,

$$
k_i^+(U)=k_i^0
  \exp\left[-\alpha\frac{F(U-E_i)}{RT}\right],
$$

$$
k_i^-(U)=k_i^0
  \exp\left[(1-\alpha)\frac{F(U-E_i)}{RT}\right],
$$

where $E_i$ is the reversible potential of that elementary step and
$\alpha$ is the transfer coefficient. At $U=E_i$, the forward and reverse
rate constants are equal in this parametrization. Their ratio changes with
potential in the direction required by the step free energy.

If the coverage vector is $\boldsymbol{\theta}$ and the stoichiometric
matrix is $\mathbf{S}$, the surface balance is

$$
\frac{d\boldsymbol{\theta}}{dt}=\mathbf{S}\mathbf{r}.
$$

At steady state,

$$
\mathbf{S}\mathbf{r}=0,
\qquad \sum_j\theta_j=1.
$$

Steady state does not mean equilibrium. At equilibrium every net elementary
rate is zero. At catalytic steady state, intermediate coverages are constant
while a nonzero cycle flux converts reactants into products.

The faradaic current follows from the electron-transfer fluxes,

$$
j_k=-F\Gamma\sum_i n_i r_i,
$$

where $\Gamma$ is the molar active-site density and $n_i$ is the number of
electrons transferred in step $i$. The negative sign denotes cathodic ORR
current in the convention used here. Mass transport is added only after the
surface kinetic current has been calculated.

The model below assumes a uniform population of independent Fe sites,
mean-field coverages, fixed activities and temperature, one associative ORR
cycle, and phenomenological standard rate constants. Its purpose is to show
how thermodynamics, kinetics, site density, and transport affect different
observables—not to claim a unique fitted FePc mechanism.


## Free-energy parameters

We use a simplified ORR free-energy path:

$$
\mathrm{\ast + O_2}
\rightarrow
\mathrm{OOH\ast}
\rightarrow
\mathrm{O\ast}
\rightarrow
\mathrm{OH\ast}
\rightarrow
\mathrm{\ast + H_2O}
$$

The final state is set to zero:

$$
G(\mathrm{\ast + H_2O}) = 0
$$

The initial state at zero potential is

$$
G(\mathrm{\ast + O_2}) = 4.92\ \mathrm{eV}
$$

because

$$
4.92 = 4 \times 1.23
$$

The adjustable thermodynamic parameters are:

$$
G_{\mathrm{OH}}
$$

$$
b_{\mathrm{OOH}}
$$

$$
b_{\mathrm{O}}
$$

with

$$
G_{\mathrm{OOH}} = G_{\mathrm{OH}} + b_{\mathrm{OOH}}
$$

and

$$
G_{\mathrm{O}} = 2(G_{\mathrm{OH}} - b_{\mathrm{O}})
$$

At potential $U$, the free-energy levels are:

$$
G_0 = 4.92 - 4U
$$

$$
G_1 = G_{\mathrm{OOH}} - 3U
$$

$$
G_2 = G_{\mathrm{O}} - 2U
$$

$$
G_3 = G_{\mathrm{OH}} - U
$$

$$
G_4 = 0
$$


In [9]:
faraday_constant_c_per_mol = 96485.0
gas_constant_j_per_mol_k = 8.314
temperature_k = 295.15
thermal_voltage_v = (gas_constant_j_per_mol_k * temperature_k / faraday_constant_c_per_mol)
equilibrium_potential_v = 1.23
total_orr_free_energy_ev = 4.0 * equilibrium_potential_v
default_params = { "G_OH": 0.82, "b_OOH": 3.03, "b_O": -0.3, "k_ads": 2000.0, "K_ads": 0.6, "k_OOH": 3.0, "k_O": 0.01, "k_OH": 0.01, "k_free": 60.0, "Gamma": 1.2e-10, }

In [11]:
def intermediate_energies(G_OH, b_OOH, b_O):
    """
    Return adsorption energies:
        G_OOH, G_O, G_OH
    """
    G_OOH = G_OH + b_OOH
    G_O = 2.0 * (G_OH - b_O)
    return [G_OOH, G_O, G_OH]


def free_energy_states(potential_v, intermediate_energies_ev):
    """
    Return the five free-energy levels:
        * + O2, OOH*, O*, OH*, * + H2O
    """
    return [ total_orr_free_energy_ev - 4.0 * potential_v, intermediate_energies_ev[0] - 3.0 * potential_v, intermediate_energies_ev[1] - 2.0 * potential_v, intermediate_energies_ev[2] - 1.0 * potential_v, 0.0, ]


def plot_at_potential(potential_v, intermediate_energies_ev, color="#0000cc", label="model"):
    """
    Plot one free-energy diagram at potential U.
    """
    states = free_energy_states(potential_v, intermediate_energies_ev)
    bar_x = [(-0.2, 0.2), (0.8, 1.2), (1.8, 2.2), (2.8, 3.2), (3.8, 4.2)]
    for i, (x0, x1) in enumerate(bar_x):
        plt.plot([x0, x1], [states[i], states[i]], c=color, linestyle="-", linewidth=3, label=f"{label}, U = {potential_v:.2f} V" if i == 0 else None)
    for i in range(4):
        plt.plot([bar_x[i][1], bar_x[i + 1][0]], [states[i], states[i + 1]], c=color, linestyle=":", linewidth=1)


def plot_free_energy_diagram(G_OH, b_OOH, b_O):
    """
    Plot free-energy diagrams at U = 1.23 V, U = U_L, and U = 0 V.
    """
    intermediate_energies_ev = intermediate_energies(G_OH, b_OOH, b_O)
    step_energies_at_U0 = [ total_orr_free_energy_ev - intermediate_energies_ev[0], intermediate_energies_ev[0] - intermediate_energies_ev[1], intermediate_energies_ev[1] - intermediate_energies_ev[2], intermediate_energies_ev[2], ]
    overpotential_v = equilibrium_potential_v - min(step_energies_at_U0)
    plt.figure()
    plot_at_potential(1.23, intermediate_energies_ev, colors[1], "Cat.")
    plot_at_potential(round(1.23 - overpotential_v, 2), intermediate_energies_ev, colors[2], "Cat.")
    plot_at_potential(0.0, intermediate_energies_ev, colors[0], "Cat.")
    plt.xlabel("Reaction coordinate")
    plt.ylabel("Free energies / eV")
    plt.xticks([0, 1, 2, 3, 4], ["*+O$_2$", "OOH*", "O*", "OH*", "*+H$_2$O"])
    plt.ylim(-1, 5.5)
    plt.legend()
    plt.show()


right_interact(plot_free_energy_diagram, G_OH=FloatSlider(value=default_params["G_OH"], min=0.5, max=1.1, step=0.01, description="G_OH / eV", continuous_update=False), b_OOH=FloatSlider(value=default_params["b_OOH"], min=2.5, max=3.5, step=0.01, description="b_OOH / eV", continuous_update=False), b_O=FloatSlider(value=default_params["b_O"], min=-0.8, max=0.2, step=0.01, description="b_O / eV", continuous_update=False))

## Steady-state coverages

The kinetic model uses the full surface cycle:

$$
1:\quad \mathrm{\ast + O_2}
\rightleftharpoons
\mathrm{O_2\ast}
$$

$$
2:\quad \mathrm{O_2\ast}
\rightleftharpoons
\mathrm{OOH\ast}
$$

$$
3:\quad \mathrm{OOH\ast}
\rightleftharpoons
\mathrm{O\ast}
$$

$$
4:\quad \mathrm{O\ast}
\rightleftharpoons
\mathrm{OH\ast}
$$

$$
5:\quad \mathrm{OH\ast}
\rightleftharpoons
\mathrm{\ast}
$$

The surface coverages are

$$
\theta_{\ast},
\theta_{\mathrm{O_2}},
\theta_{\mathrm{OOH}},
\theta_{\mathrm{O}},
\theta_{\mathrm{OH}}
$$

with the site balance

$$
\theta_{\ast}
+
\theta_{\mathrm{O_2}}
+
\theta_{\mathrm{OOH}}
+
\theta_{\mathrm{O}}
+
\theta_{\mathrm{OH}}
=
1
$$

The thermodynamic parameters are the same as in the free-energy diagram:

$$
G_{\mathrm{OOH}} = G_{\mathrm{OH}} + b_{\mathrm{OOH}}
$$

$$
G_{\mathrm{O}} = 2(G_{\mathrm{OH}} - b_{\mathrm{O}})
$$

The hidden reversible potentials used in the rate constants are calculated from the free-energy differences between neighboring states.

> **Pause and predict:** Which intermediate should dominate the surface when OH binding becomes very strong?
>
> **What to look for:** Physically meaningful coverages remain non-negative and sum to one at every potential; departures point directly to a numerical or formulation problem.


In [12]:
coverage_labels = [ "$\\theta_{\\ast}$", "$\\theta_{\\mathrm{O_2}}$", "$\\theta_{\\mathrm{OOH}}$", "$\\theta_{\\mathrm{O}}$", "$\\theta_{\\mathrm{OH}}$", ]


def thermodynamic_levels(G_OH, b_OOH, b_O, K_ads):
    """
    Energies at U = 0 for the kinetic cycle:
        *+O2, O2*, OOH*, O*, OH*, *
    """
    G_OOH = G_OH + b_OOH
    G_O = 2.0 * (G_OH - b_O)
    adsorption_free_energy_ev = -thermal_voltage_v * np.log(K_ads)
    G_O2 = total_orr_free_energy_ev + adsorption_free_energy_ev
    return {"G_O2": G_O2, "G_OOH": G_OOH, "G_O": G_O, "G_OH": G_OH}


def reversible_potentials(G_OH, b_OOH, b_O, K_ads):
    """
    Reversible potentials for electron-transfer steps 2-5.
    """
    energy_levels_ev = thermodynamic_levels(G_OH, b_OOH, b_O, K_ads)
    return { "E_OOH": energy_levels_ev["G_O2"] - energy_levels_ev["G_OOH"], "E_O": energy_levels_ev["G_OOH"] - energy_levels_ev["G_O"], "E_OH": energy_levels_ev["G_O"] - energy_levels_ev["G_OH"], "E_free": energy_levels_ev["G_OH"], }


def electrochemical_rate(k0, potential_v, equilibrium_potentials_v, alpha=0.5):
    """
    Forward and reverse rate constants for one electron-transfer step.
    """
    k_forward = k0 * np.exp(-alpha * (potential_v - equilibrium_potentials_v) / thermal_voltage_v)
    k_reverse = k0 * np.exp((1.0 - alpha) * (potential_v - equilibrium_potentials_v) / thermal_voltage_v)
    return (k_forward, k_reverse)


def rate_constants(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free):
    """
    Rate constants for the natural ORR flow:
        1 adsorption
        2 OOH formation
        3 O formation
        4 OH formation
        5 free-site regeneration
    """
    equilibrium_potentials_v = reversible_potentials(G_OH, b_OOH, b_O, K_ads)
    k1f = k_ads
    k1r = k_ads / K_ads
    k2f, k2r = electrochemical_rate(k_OOH, potential_v, equilibrium_potentials_v["E_OOH"])
    k3f, k3r = electrochemical_rate(k_O, potential_v, equilibrium_potentials_v["E_O"])
    k4f, k4r = electrochemical_rate(k_OH, potential_v, equilibrium_potentials_v["E_OH"])
    k5f, k5r = electrochemical_rate(k_free, potential_v, equilibrium_potentials_v["E_free"])
    return (k1f, k1r, k2f, k2r, k3f, k3r, k4f, k4r, k5f, k5r)


def steady_state_coverages(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free):
    """
    Solve coverages at one potential.

    Coverage order:
        *, O2*, OOH*, O*, OH*
    """
    k1f, k1r, k2f, k2r, k3f, k3r, k4f, k4r, k5f, k5r = rate_constants(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    rate_matrix = np.array([ [k1f, -(k1r + k2f), k2r, 0.0, 0.0], [0.0, k2f, -(k2r + k3f), k3r, 0.0], [0.0, 0.0, k3f, -(k3r + k4f), k4r], [k5r, 0.0, 0.0, k4f, -(k4r + k5f)], [1.0, 1.0, 1.0, 1.0, 1.0], ])
    rhs = np.array([0.0, 0.0, 0.0, 0.0, 1.0])
    return solve(rate_matrix, rhs)


def coverage_curves(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free):
    coverages = np.zeros((len(potential_grid_v), 5))
    for i, potential_v in enumerate(potential_grid_v):
        coverages[i] = steady_state_coverages(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    return coverages


def plot_coverages(G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free):
    potential_grid_v = np.linspace(1.0, 0.2, 500)
    coverages = coverage_curves(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    plt.figure()
    for i, label in enumerate(coverage_labels):
        plt.plot(potential_grid_v, coverages[:, i], label=label)
    plt.xlabel("U / V vs RHE")
    plt.ylabel("Coverage")
    plt.ylim(0, 1)
    plt.legend()
    plt.show()


right_interact(plot_coverages, G_OH=FloatSlider(value=default_params["G_OH"], min=0.5, max=1.1, step=0.01, description="G_OH", continuous_update=False), b_OOH=FloatSlider(value=default_params["b_OOH"], min=2.5, max=3.5, step=0.01, description="b_OOH", continuous_update=False), b_O=FloatSlider(value=default_params["b_O"], min=-0.8, max=0.2, step=0.01, description="b_O", continuous_update=False), k_ads=FloatLogSlider(value=default_params["k_ads"], base=10, min=1, max=5, step=0.1, description="k_ads", continuous_update=False), K_ads=FloatSlider(value=default_params["K_ads"], min=0.05, max=3.0, step=0.05, description="K_ads", continuous_update=False), k_OOH=FloatLogSlider(value=default_params["k_OOH"], base=10, min=-2, max=3, step=0.1, description="k_OOH", continuous_update=False), k_O=FloatLogSlider(value=default_params["k_O"], base=10, min=-3, max=3, step=0.1, description="k_O", continuous_update=False), k_OH=FloatLogSlider(value=default_params["k_OH"], base=10, min=-3, max=3, step=0.1, description="k_OH", continuous_update=False), k_free=FloatLogSlider(value=default_params["k_free"], base=10, min=-2, max=4, step=0.1, description="k_free", continuous_update=False))


## Kinetic current

The kinetic current is calculated from the electron-transfer steps in the surface cycle:

$$
1:\quad \mathrm{\ast + O_2}
\rightleftharpoons
\mathrm{O_2\ast}
$$

$$
2:\quad \mathrm{O_2\ast}
\rightleftharpoons
\mathrm{OOH\ast}
$$

$$
3:\quad \mathrm{OOH\ast}
\rightleftharpoons
\mathrm{O\ast}
$$

$$
4:\quad \mathrm{O\ast}
\rightleftharpoons
\mathrm{OH\ast}
$$

$$
5:\quad \mathrm{OH\ast}
\rightleftharpoons
\mathrm{\ast}
$$

Step 1 is chemical oxygen adsorption. Steps 2–5 transfer one electron each.

The kinetic current density is

$$
j_k =
-F\Gamma(r_2+r_3+r_4+r_5)
$$

Here $\Gamma$ is the active-site concentration. In the slider we use units of

$$
\mathrm{pmol\ cm^{-2}}
$$

so the code converts it as

$$
\Gamma_{\mathrm{mol}} =
\Gamma_{\mathrm{pmol}}\times 10^{-12}
$$


In [13]:
def reaction_rates(potential_v, coverages, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free):
    """
    Return reaction rates r1-r5 for the natural ORR flow.

    Coverage order:
        *, O2*, OOH*, O*, OH*
    """
    theta_free, theta_O2, theta_OOH, theta_O, theta_OH = coverages
    k1f, k1r, k2f, k2r, k3f, k3r, k4f, k4r, k5f, k5r = rate_constants(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    reaction_rate_1 = k1f * theta_free - k1r * theta_O2
    reaction_rate_2 = k2f * theta_O2 - k2r * theta_OOH
    reaction_rate_3 = k3f * theta_OOH - k3r * theta_O
    reaction_rate_4 = k4f * theta_O - k4r * theta_OH
    reaction_rate_5 = k5f * theta_OH - k5r * theta_free
    return np.array([ reaction_rate_1, reaction_rate_2, reaction_rate_3, reaction_rate_4, reaction_rate_5, ])


def kinetic_current_density(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma):
    """
    Return kinetic current density in A cm-2.

    gamma:
        Active-site concentration in mol cm-2.
    """
    coverages = steady_state_coverages(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    reaction_rates_per_site_s = reaction_rates(potential_v, coverages, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    return -faraday_constant_c_per_mol * gamma * np.sum(reaction_rates_per_site_s[1:])


def kinetic_current_curve(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma):
    current_values_a_cm2 = []
    for potential_v in potential_grid_v:
        current_a_cm2 = kinetic_current_density(potential_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma)
        current_values_a_cm2.append(current_a_cm2)
    return np.array(current_values_a_cm2)


def plot_kinetic_current(G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma):
    gamma_mol = gamma * 1e-12
    potential_grid_v = np.linspace(1.0, 0.2, 500)
    kinetic_current_a_cm2 = kinetic_current_curve(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma_mol)
    plt.figure()
    plt.plot(potential_grid_v, 1000.0 * kinetic_current_a_cm2, color=colors[0])
    plt.xlabel("U / V vs RHE")
    plt.ylabel("$j_k$ / mA cm$^{-2}$")
    plt.ylim(-10, 0)
    plt.show()


right_interact(plot_kinetic_current, G_OH=FloatSlider(value=default_params["G_OH"], min=0.5, max=1.1, step=0.01, description="G_OH", continuous_update=False), b_OOH=FloatSlider(value=default_params["b_OOH"], min=2.5, max=3.5, step=0.01, description="b_OOH", continuous_update=False), b_O=FloatSlider(value=default_params["b_O"], min=-0.8, max=0.2, step=0.01, description="b_O", continuous_update=False), k_ads=FloatLogSlider(value=default_params["k_ads"], base=10, min=1, max=5, step=0.1, description="k_ads", continuous_update=False), K_ads=FloatSlider(value=default_params["K_ads"], min=0.05, max=3.0, step=0.05, description="K_ads", continuous_update=False), k_OOH=FloatLogSlider(value=default_params["k_OOH"], base=10, min=-2, max=3, step=0.1, description="k_OOH", continuous_update=False), k_O=FloatLogSlider(value=default_params["k_O"], base=10, min=-3, max=3, step=0.1, description="k_O", continuous_update=False), k_OH=FloatLogSlider(value=default_params["k_OH"], base=10, min=-3, max=3, step=0.1, description="k_OH", continuous_update=False), k_free=FloatLogSlider(value=default_params["k_free"], base=10, min=-2, max=4, step=0.1, description="k_free", continuous_update=False), gamma=FloatSlider(value=default_params["Gamma"] / 1e-12, min=10.0, max=2000.0, step=10.0, description="gamma", continuous_update=False))


## RDE transport correction

The kinetic model gives the current density controlled only by surface reaction rates:

$$
j_k
$$

In an RRDE experiment, the measured disk current is also limited by oxygen transport to the rotating disk.

The diffusion-limited current density is estimated with the Levich equation:

$$
j_L =
-0.62 n F D_{\mathrm{O_2}}^{2/3}
\nu^{-1/6}
C_{\mathrm{O_2}}
\omega^{1/2}
$$

where $n=4$ for four-electron ORR, $D_{\mathrm{O_2}}$ is the oxygen diffusion coefficient, $\nu$ is the kinematic viscosity, $C_{\mathrm{O_2}}$ is the bulk oxygen concentration, and $\omega$ is the rotation rate in rad s$^{-1}$:

$$
\omega = \frac{2\pi\ \mathrm{rpm}}{60}
$$

The kinetic and transport limits are combined with the Koutecky-Levich relation:

$$
\frac{1}{j_{\mathrm{disk}}}
=
\frac{1}{j_k}
+
\frac{1}{j_L}
$$

The plotted current is cathodic, so it is negative.


In [14]:
from ipywidgets import Dropdown


def levich_current_density(rpm, n=4, D_O2=2e-05, nu=0.01, C_O2=1.2e-06):
    """
    Levich diffusion-limited current density in A cm-2.
    """
    omega = 2.0 * np.pi * rpm / 60.0
    return (-0.62 * n * faraday_constant_c_per_mol * D_O2 ** (2.0 / 3.0) * nu ** (-1.0 / 6.0) * C_O2 * omega**0.5)


def disk_current_density(kinetic_current_a_cm2, limiting_current_a_cm2):
    """
    Koutecky-Levich combination.

    Uses magnitudes internally and returns a cathodic current.
    """
    kinetic_current_magnitude = np.maximum(np.abs(kinetic_current_a_cm2), 1e-30)
    limiting_current_magnitude = max(abs(limiting_current_a_cm2), 1e-30)
    disk_current_magnitude = 1.0 / (1.0 / kinetic_current_magnitude + 1.0 / limiting_current_magnitude)
    return -disk_current_magnitude


def plot_rrde_disk_current(G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma, rpm):
    gamma_mol = gamma * 1e-12
    potential_grid_v = np.linspace(1.0, 0.2, 500)
    kinetic_current_a_cm2 = kinetic_current_curve(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma_mol)
    limiting_current_a_cm2 = levich_current_density(rpm)
    disk_current_a_cm2 = disk_current_density(kinetic_current_a_cm2, limiting_current_a_cm2)
    plt.figure()
    plt.plot(potential_grid_v, 1000.0 * disk_current_a_cm2, color=colors[0])
    plt.axhline(1000.0 * limiting_current_a_cm2, color=colors[4], linestyle=":", linewidth=1)
    plt.xlabel("U / V vs RHE")
    plt.ylabel("$j_{\\mathrm{disk}}$ / mA cm$^{-2}$")
    plt.ylim(-10, 0)
    plt.show()
    print(f"rpm = {rpm}")
    print(f"sqrt(rpm) = {np.sqrt(rpm):.0f}")
    print(f"j_L = {1000.0 * limiting_current_a_cm2:.3f} mA cm-2")
    print(f"most cathodic j_k = {1000.0 * np.min(kinetic_current_a_cm2):.3f} mA cm-2")


right_interact(plot_rrde_disk_current, G_OH=FloatSlider(value=default_params["G_OH"], min=0.5, max=1.1, step=0.01, description="G_OH", continuous_update=False), b_OOH=FloatSlider(value=default_params["b_OOH"], min=2.5, max=3.5, step=0.01, description="b_OOH", continuous_update=False), b_O=FloatSlider(value=default_params["b_O"], min=-0.8, max=0.2, step=0.01, description="b_O", continuous_update=False), k_ads=FloatLogSlider(value=default_params["k_ads"], base=10, min=1, max=5, step=0.1, description="k_ads", continuous_update=False), K_ads=FloatSlider(value=default_params["K_ads"], min=0.05, max=3.0, step=0.05, description="K_ads", continuous_update=False), k_OOH=FloatLogSlider(value=default_params["k_OOH"], base=10, min=-2, max=3, step=0.1, description="k_OOH", continuous_update=False), k_O=FloatLogSlider(value=default_params["k_O"], base=10, min=-3, max=3, step=0.1, description="k_O", continuous_update=False), k_OH=FloatLogSlider(value=default_params["k_OH"], base=10, min=-3, max=3, step=0.1, description="k_OH", continuous_update=False), k_free=FloatLogSlider(value=default_params["k_free"], base=10, min=-2, max=4, step=0.1, description="k_free", continuous_update=False), gamma=FloatSlider(value=default_params["Gamma"] / 1e-12, min=10.0, max=2000.0, step=10.0, description="gamma", continuous_update=False), rpm=Dropdown(options=[400, 900, 1600, 2500, 3600], value=1600, description="rpm"))


## Interactive summary dashboard

We now combine the three main outputs in one interactive dashboard.

The same parameters update:

1. the free-energy diagram,
2. the RDE disk current,
3. the steady-state coverages.

The thermodynamic sliders control the relative energies:

$$
G_{\mathrm{OH}},\quad b_{\mathrm{OOH}},\quad b_{\mathrm{O}}
$$

The kinetic sliders control the surface reaction rates:

$$
k_{\mathrm{ads}},\quad k_{\mathrm{OOH}},\quad k_{\mathrm{O}},\quad k_{\mathrm{OH}},\quad k_{\mathrm{free}}
$$

The RDE parameters are the active-site concentration

$$
\Gamma
$$

in pmol cm$^{-2}$, and the rotation rate in rpm.

The disk current is calculated from

$$
\frac{1}{j_{\mathrm{disk}}}
=
\frac{1}{j_k}
+
\frac{1}{j_L}
$$

> **Explore:** Change one slider family at a time: thermodynamics, kinetics, site density, then rotation rate.
>
> **What to look for:** Explain which output changes in position, shape, or scale and why.


In [15]:
def draw_free_energy_on_axis(ax, G_OH, b_OOH, b_O):
    intermediate_energies_ev = intermediate_energies(G_OH, b_OOH, b_O)
    step_energies_at_U0 = [ total_orr_free_energy_ev - intermediate_energies_ev[0], intermediate_energies_ev[0] - intermediate_energies_ev[1], intermediate_energies_ev[1] - intermediate_energies_ev[2], intermediate_energies_ev[2], ]
    overpotential_v = equilibrium_potential_v - min(step_energies_at_U0)
    U_values = [1.23, round(1.23 - overpotential_v, 2), 0.0]
    U_colors = [colors[1], colors[2], colors[0]]
    bar_x = [(-0.2, 0.2), (0.8, 1.2), (1.8, 2.2), (2.8, 3.2), (3.8, 4.2)]
    for potential_v, color in zip(U_values, U_colors):
        states = free_energy_states(potential_v, intermediate_energies_ev)
        for i, (x0, x1) in enumerate(bar_x):
            ax.plot([x0, x1], [states[i], states[i]], c=color, linestyle="-", linewidth=3, label=f"U = {potential_v:.2f} V" if i == 0 else None)
        for i in range(4):
            ax.plot([bar_x[i][1], bar_x[i + 1][0]], [states[i], states[i + 1]], c=color, linestyle=":", linewidth=1)
    ax.set_xlabel("Reaction coordinate")
    ax.set_ylabel("Free energies / eV")
    ax.set_xticks([0, 1, 2, 3, 4])
    ax.set_xticklabels(["*+O$_2$", "OOH*", "O*", "OH*", "*+H$_2$O"])
    ax.set_ylim(-1, 5.5)
    ax.legend()
    ax.set_title("Free-energy diagram")


def half_wave_potential(x, y):
    y_start = y[0]
    y_end = y[-1]
    y_half = 0.5 * (y_start + y_end)
    x_half = None
    for i in range(len(x) - 1):
        y0 = y[i]
        y1 = y[i + 1]
        if (y0 - y_half) * (y1 - y_half) <= 0:
            if y1 == y0:
                x_half = 0.5 * (x[i] + x[i + 1])
            else:
                x_half = x[i] + (y_half - y0) * (x[i + 1] - x[i]) / (y1 - y0)
            break
    return (x_half, y_half)


def plot_summary_dashboard(G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma, rpm):
    gamma_mol = gamma * 1e-12
    potential_grid_v = np.linspace(1.0, 0.2, 500)
    coverages = coverage_curves(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free)
    kinetic_current_a_cm2 = kinetic_current_curve(potential_grid_v, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma_mol)
    limiting_current_a_cm2 = levich_current_density(rpm)
    disk_current_a_cm2 = disk_current_density(kinetic_current_a_cm2, limiting_current_a_cm2)
    fig, axes = plt.subplots(1, 3, figsize=(3 * figure_width_inches, figure_width_inches))
    draw_free_energy_on_axis(axes[0], G_OH, b_OOH, b_O)
    disk_current_ma_cm2 = 1000.0 * disk_current_a_cm2
    half_wave_potential_v, half_wave_current_ma_cm2 = half_wave_potential(potential_grid_v, disk_current_ma_cm2)
    axes[1].plot(potential_grid_v, disk_current_ma_cm2, color=colors[0])
    axes[1].axhline(1000.0 * limiting_current_a_cm2, color=colors[4], linestyle=":", linewidth=1)
    if half_wave_potential_v is not None:
        axes[1].axhline(half_wave_current_ma_cm2, color=colors[3], linestyle="--", linewidth=1)
        axes[1].axvline(half_wave_potential_v, color=colors[3], linestyle="--", linewidth=1)
        axes[1].plot(half_wave_potential_v, half_wave_current_ma_cm2, marker="o", color=colors[3])
        axes[1].text(half_wave_potential_v, half_wave_current_ma_cm2, f"  E$_{{1/2}}$ = {half_wave_potential_v:.3f} V", va="bottom", ha="left")
    axes[1].set_xlabel("U / V vs RHE")
    axes[1].set_ylabel("$j_{\\mathrm{disk}}$ / mA cm$^{-2}$")
    axes[1].set_ylim(-10, 0)
    axes[1].set_title(f"RRDE disk current, {rpm} rpm")
    for i, label in enumerate(coverage_labels):
        axes[2].plot(potential_grid_v, coverages[:, i], label=label)
    axes[2].set_xlabel("U / V vs RHE")
    axes[2].set_ylabel("Coverage")
    axes[2].set_ylim(0, 1)
    axes[2].legend()
    axes[2].set_title("Surface coverages")
    plt.tight_layout()
    plt.show()


right_interact(plot_summary_dashboard, G_OH=FloatSlider(value=default_params["G_OH"], min=0.5, max=1.1, step=0.01, description="G_OH", continuous_update=False), b_OOH=FloatSlider(value=default_params["b_OOH"], min=2.5, max=3.5, step=0.01, description="b_OOH", continuous_update=False), b_O=FloatSlider(value=default_params["b_O"], min=-0.8, max=0.2, step=0.01, description="b_O", continuous_update=False), k_ads=FloatLogSlider(value=default_params["k_ads"], base=10, min=1, max=5, step=0.1, description="k_ads", continuous_update=False), K_ads=FloatSlider(value=default_params["K_ads"], min=0.05, max=3.0, step=0.05, description="K_ads", continuous_update=False), k_OOH=FloatLogSlider(value=default_params["k_OOH"], base=10, min=-2, max=3, step=0.1, description="k_OOH", continuous_update=False), k_O=FloatLogSlider(value=default_params["k_O"], base=10, min=-3, max=3, step=0.1, description="k_O", continuous_update=False), k_OH=FloatLogSlider(value=default_params["k_OH"], base=10, min=-3, max=3, step=0.1, description="k_OH", continuous_update=False), k_free=FloatLogSlider(value=default_params["k_free"], base=10, min=-2, max=4, step=0.1, description="k_free", continuous_update=False), gamma=FloatSlider(value=default_params["Gamma"] / 1e-12, min=10.0, max=2000.0, step=10.0, description="gamma", continuous_update=False), rpm=Dropdown(options=[400, 900, 1600, 2500, 3600], value=1600, description="rpm"))


## Summary

By working through this notebook, you have practiced how to:

- Translate an ORR free-energy landscape into reversible elementary rates.
- Solve steady-state surface coverages with a site-balance constraint.
- Convert elementary rates into a kinetic current density.
- Combine kinetic and Levich currents with the Koutecky–Levich relation.
- Use one dashboard to separate thermodynamic, kinetic, site-density, and
  mass-transport effects.

> **Key takeaway:** Thermodynamics sets equilibrium tendencies, kinetics
> sets surface rates and steady-state populations, and mass transport limits
> the current that an experiment can measure.

## Seminar exercises

1. Insert the FePc free energies from Seminar 1 and identify which slider
   parameters they replace.
2. Change only one standard rate constant and explain whether the free-energy
   diagram, coverage, kinetic current, or Levich plateau should move.
3. Compare equilibrium coverage from Seminar 2 with the steady-state site
   balance here.  State the physical reason they need not agree.
4. Increase rotation rate and active-site density separately.  Identify
   which current regime responds to each change.
